# 01 - Data Collection

This notebook collects public Old School RuneScape Hiscores data for selected players.

The collected data will later be used for unsupervised machine learning, mainly clustering-based player segmentation.

In [1]:
import requests
import pandas as pd
import time
from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_DIR, PROCESSED_DATA_DIR

(WindowsPath('C:/Projects/osrs-player-segmentation/data/raw'),
 WindowsPath('C:/Projects/osrs-player-segmentation/data/processed'))

In [3]:
SKILLS = [
    "overall",
    "attack",
    "defence",
    "strength",
    "hitpoints",
    "ranged",
    "prayer",
    "magic",
    "cooking",
    "woodcutting",
    "fletching",
    "fishing",
    "firemaking",
    "crafting",
    "smithing",
    "mining",
    "herblore",
    "agility",
    "thieving",
    "slayer",
    "farming",
    "runecraft",
    "hunter",
    "construction",
    "sailing"
]

In [4]:
player_names = [
    "Zezima",
    "eszcape",
    "Lynx Titan",
    "Hey Jase",
    "Woox",
    "B0aty",
    "Alkan",
    "Settled",
    "A Cold One",
    "Mammal",
    "SoloMission"
]

In [5]:
def fetch_hiscores(player_name):
    """
    Fetch raw OSRS Hiscores data for a single player.
    Returns raw text response or None if the request fails.
    """
    url = "https://secure.runescape.com/m=hiscore_oldschool/index_lite.ws"
    params = {"player": player_name}

    try:
        response = requests.get(url, params=params, timeout=10)
        
        if response.status_code == 200:
            return response.text
        
        print(f"Failed to fetch {player_name}: HTTP {response.status_code}")
        return None
    
    except requests.RequestException as error:
        print(f"Request error for {player_name}: {error}")
        return None

In [6]:
def parse_skill_data(player_name, raw_text):
    """
    Parse the skill section of OSRS Hiscores data.

    The first 24 rows represent:
    rank, level, experience for each skill.
    """
    lines = raw_text.strip().splitlines()
    skill_lines = lines[:len(SKILLS)]

    player_data = {"player": player_name}

    for skill_name, line in zip(SKILLS, skill_lines):
        values = line.split(",")

        if len(values) < 3:
            continue

        rank, level, xp = values[:3]

        player_data[f"{skill_name}_rank"] = int(rank)
        player_data[f"{skill_name}_level"] = int(level)
        player_data[f"{skill_name}_xp"] = int(xp)

    return player_data

In [7]:
records = []

for player_name in player_names:
    print(f"Fetching: {player_name}")
    
    raw_text = fetch_hiscores(player_name)
    
    if raw_text is None:
        continue
    
    parsed_data = parse_skill_data(player_name, raw_text)
    records.append(parsed_data)
    
    time.sleep(1)

df = pd.DataFrame(records)
df.head()

Fetching: Zezima
Fetching: eszcape
Fetching: Lynx Titan
Fetching: Hey Jase
Fetching: Woox
Fetching: B0aty
Fetching: Alkan
Fetching: Settled
Fetching: A Cold One
Fetching: Mammal
Failed to fetch Mammal: HTTP 404
Fetching: SoloMission


,player,overall_rank,overall_level,overall_xp,attack_rank,attack_level,attack_xp,defence_rank,defence_level,defence_xp,...,runecraft_xp,hunter_rank,hunter_level,hunter_xp,construction_rank,construction_level,construction_xp,sailing_rank,sailing_level,sailing_xp
0,Zezima,1571605,1466,27957906,1556666,76,1343681,1420448,76,1342072,...,57263,1917011,45,62118,1895720,45,66561,-1,1,0
1,eszcape,317297,2107,201501042,251680,99,14852643,293228,99,13541188,...,1455752,478810,82,2505555,600507,83,2710608,330098,60,297498
2,Lynx Titan,103734,2278,4600000000,15,99,200000000,28,99,200000000,...,200000000,3,99,200000000,4,99,200000000,-1,1,0
3,Hey Jase,103735,2278,4600000000,19,99,200000000,58,99,200000000,...,200000000,34,99,200000000,10,99,200000000,-1,1,0
4,Woox,66972,2340,507712649,58399,99,26061798,33458,99,25478462,...,14386618,42409,99,15556190,32651,99,13400250,304792,63,394149


In [8]:
df.shape

(10, 76)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 76 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   player              10 non-null     object
 1   overall_rank        10 non-null     int64 
 2   overall_level       10 non-null     int64 
 3   overall_xp          10 non-null     int64 
 4   attack_rank         10 non-null     int64 
 5   attack_level        10 non-null     int64 
 6   attack_xp           10 non-null     int64 
 7   defence_rank        10 non-null     int64 
 8   defence_level       10 non-null     int64 
 9   defence_xp          10 non-null     int64 
 10  strength_rank       10 non-null     int64 
 11  strength_level      10 non-null     int64 
 12  strength_xp         10 non-null     int64 
 13  hitpoints_rank      10 non-null     int64 
 14  hitpoints_level     10 non-null     int64 
 15  hitpoints_xp        10 non-null     int64 
 16  ranged_rank         10 non-nu

In [10]:
df.head()

,player,overall_rank,overall_level,overall_xp,attack_rank,attack_level,attack_xp,defence_rank,defence_level,defence_xp,...,runecraft_xp,hunter_rank,hunter_level,hunter_xp,construction_rank,construction_level,construction_xp,sailing_rank,sailing_level,sailing_xp
0,Zezima,1571605,1466,27957906,1556666,76,1343681,1420448,76,1342072,...,57263,1917011,45,62118,1895720,45,66561,-1,1,0
1,eszcape,317297,2107,201501042,251680,99,14852643,293228,99,13541188,...,1455752,478810,82,2505555,600507,83,2710608,330098,60,297498
2,Lynx Titan,103734,2278,4600000000,15,99,200000000,28,99,200000000,...,200000000,3,99,200000000,4,99,200000000,-1,1,0
3,Hey Jase,103735,2278,4600000000,19,99,200000000,58,99,200000000,...,200000000,34,99,200000000,10,99,200000000,-1,1,0
4,Woox,66972,2340,507712649,58399,99,26061798,33458,99,25478462,...,14386618,42409,99,15556190,32651,99,13400250,304792,63,394149


In [11]:
output_path = PROCESSED_DATA_DIR / "osrs_hiscores_sample.csv"

df.to_csv(output_path, index=False, encoding="utf-8")

output_path

WindowsPath('C:/Projects/osrs-player-segmentation/data/processed/osrs_hiscores_sample.csv')

## Summary

In this notebook, we collected public OSRS Hiscores data for a small sample of players.

The dataset contains skill-related rank, level and experience values.

The next step is to clean the dataset and prepare it for clustering.